In [43]:
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
df = pd.read_csv(r"'FC26_20250921.csv'")
df.head()

C:\Users\Onabanjo Daniel\AppData\Local\Temp\ipykernel_16616\2741521708.py:1: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\Onabanjo Daniel\Desktop\Stuff\Career\Personal Projects\FC Project\FC26_20250921.csv")


,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,...,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url
0,252371,/player/252371/jude-bellingham/260004/,26,4,2025-09-19,J. Bellingham,Jude Victor William Bellingham,"CAM, CM",90,94,...,85+3,85+3,83+3,82+3,81+3,81+3,81+3,82+3,18+3,https://cdn.sofifa.net/players/252/371/26_120.png
1,239053,/player/239053/federico-valverde/260004/,26,4,2025-09-19,F. Valverde,Federico Santiago Valverde Dipetta,"CM, CDM, RB",89,90,...,87+3,87+3,86+3,86+3,83+3,83+3,83+3,86+3,18+3,https://cdn.sofifa.net/players/239/053/26_120.png
2,212622,/player/212622/joshua-kimmich/260004/,26,4,2025-09-19,J. Kimmich,Joshua Walter Kimmich,"CDM, RB, CM",89,89,...,87+2,87+2,86+3,85+3,82+3,82+3,82+3,85+3,21+3,https://cdn.sofifa.net/players/212/622/26_120.png
3,235212,/player/235212/achraf-hakimi/260004/,26,4,2025-09-19,A. Hakimi,Achraf Hakimi Mouhأشرف حكيمي,"RB, RM",89,90,...,83+3,83+3,86+3,86+3,81+3,81+3,81+3,86+3,17+3,https://cdn.sofifa.net/players/235/212/26_120.png
4,224232,/player/224232/nicolo-barella/260004/,26,4,2025-09-19,N. Barella,Nicolò Barella,CM,87,87,...,85+2,85+2,84+3,83+3,80+3,80+3,80+3,83+3,19+3,https://cdn.sofifa.net/players/224/232/26_120.png


### Version 1 (V1) Project
Build a simple FC 26 Player Finder in Python.

Users will be able to filter by: position, age, potential, overall, value, club, league, height, nationality, weak foot, and skill moves.

For the initial position filter, a player matches if the selected position appears anywhere in player_positions. Later, we can add a primary-position-only option.

The first version will focus on filtering, with ranking, player similarity, scouting logic, optimization, and other features added later.

### Initial Inspection Summary
- Each player has a unique player_id; player names are not guaranteed to be unique.
- fifa_version, fifa_update, and fifa_update_date are constant across the dataset: 26, 4, 2025-09-19.
- player_positions lists the positions a player can play, with the first position being primary. Example: CAM, CM.
- Position-specific columns such as cam, cm, rw, cb contain the player's rating in that position.
- No missing values were found in the main V1 fields: player_positions, overall, potential, age, value_eur, height_cm, club_name, league_name, nationality_name, weak_foot, and skill_moves. But some value_eur values were 0
- These V1 numeric fields are stored as int64; categorical fields are strings.
- league_level ranges from 1–4, with 1 being the highest level.
- 89 players are missing club/league information such as league_name, club_name, league_id, and league_level.
- 1,414 players have no club_joined_date.
- National-team fields have many missing values because they apply only to players in the national-team setup at the time; nation_team_id, nation_position, and nation_jersey_number together account for 17,677 missing values.
- international_reputation ranges from 1–5, with 5 apparently representing the highest reputation.
- work_rate is completely empty.
- body_type contains several categories, but some appear inconsistently labelled.
- release_clause_eur has 1,434 missing values, which can reasonably represent players without release clauses.
- player_tags and player_traits have many missing values because these characteristics are not assigned to every player.
Goalkeepers have missing outfield attributes (pace, shooting, passing, dribbling, defending, physic), while outfield players have missing goalkeeper attributes. These should be treated as not applicable, rather than automatically imputed.
- Date fields such as dob are currently stored as strings.
- Some leagues have the same name e.g. English and Russian Premier League, so the league name is not a reliable unique identifier

In [45]:
df['player_positions_list'] = df['player_positions'].str.split(', ')
cm_players = df[(df['player_positions_list'].apply(lambda x: 'CM' in x ))]
cm_players

,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,...,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url,player_positions_list
0,252371,/player/252371/jude-bellingham/260004/,26,4,2025-09-19,J. Bellingham,Jude Victor William Bellingham,"CAM, CM",90,94,...,85+3,83+3,82+3,81+3,81+3,81+3,82+3,18+3,https://cdn.sofifa.net/players/252/371/26_120.png,"[CAM, CM]"
1,239053,/player/239053/federico-valverde/260004/,26,4,2025-09-19,F. Valverde,Federico Santiago Valverde Dipetta,"CM, CDM, RB",89,90,...,87+3,86+3,86+3,83+3,83+3,83+3,86+3,18+3,https://cdn.sofifa.net/players/239/053/26_120.png,"[CM, CDM, RB]"
2,212622,/player/212622/joshua-kimmich/260004/,26,4,2025-09-19,J. Kimmich,Joshua Walter Kimmich,"CDM, RB, CM",89,89,...,87+2,86+3,85+3,82+3,82+3,82+3,85+3,21+3,https://cdn.sofifa.net/players/212/622/26_120.png,"[CDM, RB, CM]"
4,224232,/player/224232/nicolo-barella/260004/,26,4,2025-09-19,N. Barella,Nicolò Barella,CM,87,87,...,85+2,84+3,83+3,80+3,80+3,80+3,83+3,19+3,https://cdn.sofifa.net/players/224/232/26_120.png,[CM]
5,208128,/player/208128/hakan-calhanoglu/260004/,26,4,2025-09-19,H. Çalhanoğlu,Hakan Çalhanoğlu,"CDM, CM",86,86,...,84+2,82+3,81+3,79+3,79+3,79+3,81+3,18+3,https://cdn.sofifa.net/players/208/128/26_120.png,"[CDM, CM]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16233,276630,/player/276630/franklin-nazareth/260004/,26,4,2025-09-19,F. Nazareth,Franklin Robin Nazareth,"CDM, CM, CAM",49,60,...,48+2,46+2,46+2,49+2,49+2,49+2,46+2,13+2,https://cdn.sofifa.net/players/276/630/26_120.png,"[CDM, CM, CAM]"
16264,73099,/player/73099/alex-nunes/260004/,26,4,2025-09-19,A. Nunes,Alex Nunes,"CAM, CM",50,73,...,41+2,41+2,40+2,37+2,37+2,37+2,40+2,11+2,https://cdn.sofifa.net/players/073/099/26_120.png,"[CAM, CM]"
16462,264056,/player/264056/junxian-liu/260004/,26,4,2025-09-19,Liu Junxian,Liu Junxian刘俊贤,"ST, CM",49,56,...,29+2,32+2,31+2,27+2,27+2,27+2,31+2,15+2,https://cdn.sofifa.net/players/264/056/26_120.png,"[ST, CM]"
16553,71007,/player/71007/jun-yeong-choi/260004/,26,4,2025-09-19,Choi Jun Yeong,Jun-yeong Choi최준영,"CB, CM",51,61,...,46+2,46+2,48+2,51+2,51+2,51+2,48+2,12+2,https://cdn.sofifa.net/players/071/007/26_120.png,"[CB, CM]"


In [46]:
fltr = df[(df['player_positions_list'].apply(lambda x: 'RW' in x )) & (df['age'] <=17)& (df['potential'] > 80) ]
fltr

,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,...,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url,player_positions_list
1104,277643,/player/277643/lamine-yamal-nasraoui-ebana/260...,26,4,2025-09-19,Lamine Yamal,Lamine Yamal Nasraoui Ebanaلامين يامال نصراوي ...,"RM, RW",89,95,...,56+3,62+3,55+3,39+3,39+3,39+3,55+3,18+3,https://cdn.sofifa.net/players/277/643/26_120.png,"[RM, RW]"
1153,279173,/player/279173/franco-mastantuono/260004/,26,4,2025-09-19,F. Mastantuono,Franco Mastantuono,"CAM, RW, ST",77,88,...,65+2,66+2,63+2,56+2,56+2,56+2,63+2,16+2,https://cdn.sofifa.net/players/279/173/26_120.png,"[CAM, RW, ST]"
5630,74364,/player/74364/andre-garcia/260004/,26,4,2025-09-19,A. Garcia,Andre Garcia,"LB, RW, LW, RM",63,82,...,57+2,61+2,61+2,57+2,57+2,57+2,61+2,17+2,https://cdn.sofifa.net/players/074/364/26_120.png,"[LB, RW, LW, RM]"
7776,80816,/player/80816/bradley-burrowes/260004/,26,4,2025-09-19,B. Burrowes,Bradley Burrowes,"RM, ST, RW",65,82,...,47+2,52+2,50+2,44+2,44+2,44+2,50+2,16+2,https://cdn.sofifa.net/players/080/816/26_120.png,"[RM, ST, RW]"
7803,70497,/player/70497/konstantinos-karetsas/260004/,26,4,2025-09-19,K. Karetsas,Konstantinos Karetsas,"CAM, RW, CM",70,86,...,47+2,51+2,46+2,33+2,33+2,33+2,46+2,14+2,https://cdn.sofifa.net/players/070/497/26_120.png,"[CAM, RW, CM]"
8028,74626,/player/74626/bruno-durdov/260004/,26,4,2025-09-19,B. Durdov,Bruno Durdov,"RW, RM",64,84,...,47+2,51+2,48+2,42+2,42+2,42+2,48+2,13+2,https://cdn.sofifa.net/players/074/626/26_120.png,"[RW, RM]"
10082,77146,/player/77146/maxloren-castro/260004/,26,4,2025-09-19,M. Castro,Maxloren Sannoe Castro Rufino,"LW, LM, RW",68,81,...,50+2,55+2,52+2,43+2,43+2,43+2,52+2,14+2,https://cdn.sofifa.net/players/077/146/26_120.png,"[LW, LM, RW]"
10537,74449,/player/74449/ibrahim-mbaye/260004/,26,4,2025-09-19,I. Mbaye,Ibrahim Mbaye,"RW, LW, RM",68,83,...,44+2,49+2,45+2,36+2,36+2,36+2,45+2,17+2,https://cdn.sofifa.net/players/074/449/26_120.png,"[RW, LW, RM]"
11980,77349,/player/77349/sandro-miguel-rodrigues-vidigal/...,26,4,2025-09-19,Sandro Vidigal,Sandro Miguel Rodrigues Vidigal,"LW, RW, CAM, LM",65,82,...,43+2,49+2,46+2,36+2,36+2,36+2,46+2,14+2,https://cdn.sofifa.net/players/077/349/26_120.png,"[LW, RW, CAM, LM]"
12526,75156,/player/75156/antonio-fernandez-casino/260004/,26,4,2025-09-19,Toni Fernández,Antonio Fernández Casino,"RW, RM",64,83,...,42+2,46+2,43+2,35+2,35+2,35+2,43+2,13+2,https://cdn.sofifa.net/players/075/156/26_120.png,"[RW, RM]"


In [47]:
def get_valid_entries(parameter, minimum, maximum):
    while True:
        try:
            entry = input(f'{parameter}: ').strip()
            if entry == "":
                return None
            else:
                entry = int(entry)
                if (entry > maximum) or (entry < minimum):
                    print('Enter a valid entry') 
                else:
                    break
        except ValueError:
            print('Enter a valid entry')
    return entry

def get_valid_float_entries(parameter, minimum, maximum):
    while True:
        try:
            entry = input(f'{parameter}: ').strip()
            if entry == "":
                return None
            else:
                entry = float(entry)
                if (entry > maximum) or (entry < minimum):
                    print('Enter a valid entry') 
                else:
                    break
        except ValueError:
            print('Enter a valid entry')
    return entry

In [80]:
position = input('Position: ').upper()

league = input('League: ').title()

club = input('Club: ').title()

nationality = input('Nationality: ').title()

min_age = get_valid_entries('Min Age: ', 1, 99)
max_age = get_valid_entries('Max Age: ', 1, 99)

min_overall = get_valid_entries('Min Overall: ', 1, 99)
max_overall = get_valid_entries('Max Overall: ', 1, 99)

min_potential = get_valid_entries('Min Potential: ', 1, 99)
max_potential = get_valid_entries('Max Potential: ', 1, 99)

weak_foot = get_valid_entries('Weak Foot: ', 1, 5)
skill_moves = get_valid_entries('Skill Moves: ', 1, 5)

min_height_cm = get_valid_entries('Min Height: ', 1, 300)
max_height_cm = get_valid_entries('Max Height: ', 1, 300)

min_val_eur = get_valid_float_entries('Min Value (million euros): ', 0, 1000)
max_val_eur = get_valid_float_entries('Max Value (million euros): ', 0, 1000)

Position:  rw
League:  
Club:  
Nationality:  
Min Age: :  15
Max Age: :  20
Min Overall: :  11
Max Overall: :  99
Min Potential: :  85
Max Potential: :  99
Weak Foot: :  
Skill Moves: :  
Min Height: :  
Max Height: :  
Min Value (million euros): :  
Max Value (million euros): :  


In [81]:
filtered = df

if position:
    filtered = filtered[filtered['player_positions_list'].apply(lambda x: position in x)]

if min_age is not None:
    filtered = filtered[filtered['age'] >= min_age]
if max_age is not None:
    filtered = filtered[filtered['age'] <= max_age]

if min_overall is not None:
    filtered = filtered[filtered['overall'] >= min_overall]
if max_overall is not None:
    filtered = filtered[filtered['overall'] <= max_overall]

if min_potential is not None:
    filtered = filtered[filtered['potential'] >= min_potential]
if max_potential is not None:
    filtered = filtered[filtered['potential'] <= max_potential]

if weak_foot is not None:
    filtered = filtered[filtered['weak_foot'] >= weak_foot]

if skill_moves is not None:
    filtered = filtered[filtered['skill_moves'] >= skill_moves]

if min_height_cm is not None:
    filtered = filtered[filtered['height_cm'] >= min_height_cm]
if max_height_cm is not None:
    filtered = filtered[filtered['height_cm'] <= max_height_cm]

if min_val_eur is not None:
    filtered = filtered[filtered['value_eur'] >= (min_val_eur*1000000)]
if max_val_eur is not None:
    filtered = filtered[filtered['value_eur'] <= (max_val_eur*1000000)]

if league:
    filtered = filtered[filtered['league_name'] == league]
    clubs = filtered['club_name'].dropna().unique().tolist()
    if club:
        if club in clubs:
            filtered = filtered[filtered['club_name'] == club]
        else:
            print('That club is not in the selected league')
else:
    if club:
       filtered = filtered[filtered['club_name'] == club]

if nationality:
    filtered = filtered[filtered['nationality_name'] == nationality]
    
results = filtered[['short_name', 'age', 'player_positions', 'overall', 'potential', 'club_name', 'nationality_name', 'value_eur']]
players_found = results.shape[0]
print(f'Found {players_found} players')
if players_found == 0:
    print('No Players Match Your Conditions')
else:
    display(results)

Found 21 players


,short_name,age,player_positions,overall,potential,club_name,nationality_name,value_eur
155,D. Doué,20,"RW, LW, CM, RM",85,91,Paris Saint-Germain,France,83500000
732,A. Güler,20,"RM, CAM, RW",81,89,Real Madrid,Türkiye,56500000
1104,Lamine Yamal,17,"RM, RW",89,95,FC Barcelona,Spain,147000000
1153,F. Mastantuono,17,"CAM, RW, ST",77,88,Real Madrid,Argentina,22000000
1423,Estêvão,18,"RM, CAM, RW",78,89,Chelsea,Brazil,29500000
1696,Roger Fernandes,19,"RM, LM, RW",76,85,Al Ittihad,Portugal,16000000
2047,Endrick,18,"ST, RW",77,91,Real Madrid,Brazil,24500000
2111,E. Nwaneri,18,"RW, CM, RM",76,87,Arsenal,England,16000000
2317,Y. Minteh,20,"RM, RW",77,85,Brighton & Hove Albion,Gambia,23000000
2768,Geovany Quenda,18,"RM, LM, LW, RW",76,88,Sporting CP,Portugal,17500000


In [99]:
df.query('value_eur==0')['league_name'].unique().tolist()

[nan,
 'Pro League',
 'La Liga',
 'Liga Profesional de Fútbol',
 'Ekstraklasa',
 'Liga 1',
 'Ligue 1',
 'Serie A',
 'Super League',
 'Major League Soccer',
 'División Profesional',
 'Primeira Liga',
 'Primera División']